# 🎯 Notebook 5: Concurrency Limits & Production Practice

The first four notebooks answered **"how many requests per second?"**.
This one answers the rest of the questions you'll actually get asked in a
design review:

1. **Concurrency limiting** — *"how many *in-flight* requests?"* — a
   different problem from rate limiting, solved with a **semaphore**.
2. **Cost / weight-based limits** — not all requests are equal (a
   `GET /search?q=*` is 100× a `GET /ping`).
3. **Where to put the limiter** — edge (CDN / API gateway) vs service.
4. **Fail-open vs fail-closed** — what happens when the limiter itself
   breaks.
5. **Choosing the limit key** — IP, user, API key, tenant — and the
   traps (NAT, IPv6, shared CI runners, spoofed headers).


## 🛠️ Setup

```bash
cd 04-patterns/rate-limiting-and-throttling
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## 1️⃣ Concurrency limiting (a.k.a. in-flight cap)

Rate limiting caps **arrivals** ("100 req/sec"). But a slow endpoint can
still be overloaded at a low arrival rate: 50 req/sec **each taking 10s**
means **500 in-flight** — that exhausts DB connections, memory, threads.

The fix is **concurrency limiting**: *"at most N requests processing at
the same time"*. It's just a **semaphore**. Any request above N waits
(or is rejected with `503 Service Unavailable`).

> 💡 Rule of thumb: use **both**. Rate limit for fairness/abuse, concurrency
> limit to protect finite resources (DB pool size, worker threads).


### ❌ Bad: no concurrency cap — slow endpoint kills you

Simulate 30 requests, each sleeping 1s, hitting a server with no cap.

In [ ]:
import time, threading

in_flight = 0
max_seen = 0
lock = threading.Lock()

def slow_handler():
    global in_flight, max_seen
    with lock:
        in_flight += 1
        max_seen = max(max_seen, in_flight)
    time.sleep(1.0)   # pretend the endpoint is slow
    with lock:
        in_flight -= 1

threads = [threading.Thread(target=slow_handler) for _ in range(30)]
t0 = time.monotonic()
for t in threads: t.start()
for t in threads: t.join()
print(f"peak in-flight: {max_seen}  (all 30 ran at once)")
print(f"elapsed: {time.monotonic() - t0:.2f}s")

### ✅ Better: cap with a semaphore

A semaphore counts how many "slots" are in use. Acquire before work,
release after. Extras either **wait** (throttle) or get **rejected**
(reject = return `503` with `Retry-After`).

In [ ]:
from threading import Semaphore

class ConcurrencyLimiter:
    def __init__(self, max_in_flight):
        self.sem = Semaphore(max_in_flight)

    def try_acquire(self, timeout=0):
        """timeout=0    -> reject immediately when full (fast-fail, return 503)
        timeout>0    -> wait up to that long, then give up (throttle)
        timeout=None -> wait forever (only safe if the caller has its own deadline)"""
        if timeout == 0:
            return self.sem.acquire(blocking=False)
        return self.sem.acquire(timeout=timeout)

    def release(self):
        self.sem.release()

limiter = ConcurrencyLimiter(max_in_flight=5)
rejected = accepted = 0
max_seen = in_flight = 0
lock = threading.Lock()

def handler():
    global in_flight, max_seen, accepted, rejected
    if not limiter.try_acquire():
        with lock: rejected += 1
        return
    try:
        with lock:
            in_flight += 1; max_seen = max(max_seen, in_flight); accepted += 1
        time.sleep(1.0)
    finally:
        with lock: in_flight -= 1
        limiter.release()

threads = [threading.Thread(target=handler) for _ in range(30)]
for t in threads: t.start()
for t in threads: t.join()
print(f"accepted: {accepted}, rejected: {rejected}, peak in-flight: {max_seen}")

## 2️⃣ Cost-based (weighted) rate limiting

Real APIs have cheap and expensive endpoints. GitHub's GraphQL API is
famous for this: each query has a **cost** (`1`, `5`, `50`…) and your
budget is measured in **points**, not requests.

Our token bucket already supports this — the `n` argument is the cost.


In [ ]:
class TokenBucket:
    def __init__(self, rate, capacity):
        self.rate = rate; self.capacity = capacity
        self.tokens = capacity; self.last = time.monotonic()
    def allow(self, n=1):
        now = time.monotonic()
        self.tokens = min(self.capacity, self.tokens + (now - self.last) * self.rate)
        self.last = now
        if self.tokens >= n:
            self.tokens -= n; return True
        return False

# Budget: 10 points/sec, 50 point burst
budget = TokenBucket(rate=10, capacity=50)

COSTS = {
    "GET /ping":          1,    # cheap
    "GET /user/me":       2,    # small read
    "GET /search?q=...": 20,    # expensive
    "POST /export":      40,    # very expensive
}

for endpoint, cost in COSTS.items():
    ok = budget.allow(cost)
    print(f"{endpoint:<22} cost={cost:<3} → {'allowed' if ok else 'REJECTED'}  tokens_left={budget.tokens:.1f}")

## 3️⃣ Where should the limiter live?

```
  client → [CDN/WAF] → [API gateway] → [service] → [DB]
             ↑             ↑              ↑
      cheap global    per-route      business-logic
      abuse block    per-user cap    ("max 3 login
                                       attempts")
```

- **Edge (CDN / WAF — Cloudflare, AWS WAF)**: cheapest to reject — you
  don't even pay for compute. Good for coarse limits ("100 req/sec per IP
  globally"). Bad at knowing *who the user is*.
- **API gateway (Kong, NGINX, Envoy, AWS API Gateway)**: knows routes and
  API keys. Best place for per-tenant quotas.
- **Service code**: knows business context ("login", "password reset") and
  can apply **different** limits to different operations.

> 💡 You usually want **defense in depth**: a generous limit at the edge,
> a tighter per-user limit at the gateway, and specific business limits
> inside the service (e.g., login attempts).


## 4️⃣ Fail-open vs fail-closed

Your Redis goes down. What should the limiter do?

| Strategy | Behavior on outage | Risk |
|---|---|---|
| **Fail-open** (allow) | Treat everything as allowed | You lose the limiter — abusers win. **But the service stays up.** |
| **Fail-closed** (deny) | Treat everything as denied | 100% outage propagates. |
| **Fallback to local** | Use a per-server in-memory limiter during outage | Imperfect but graceful. |

Most user-facing APIs **fail open** (availability > perfect enforcement).
Auth / anti-abuse paths sometimes **fail closed** (security > availability).


In [ ]:
class ResilientLimiter:
    def __init__(self, shared, local_fallback, mode="fail_open"):
        self.shared = shared
        self.local = local_fallback
        self.mode = mode

    def allow(self, key):
        try:
            return self.shared.allow(key)
        except Exception as e:
            # log it (in real life: metric + alert)
            print(f"[WARN] shared store down: {e} — falling back to local")
            if self.mode == "fail_open":
                return True
            if self.mode == "fail_closed":
                return False
            return self.local.allow(key)   # "fail_local"

class BrokenShared:
    def allow(self, key): raise RuntimeError("redis timeout")

class LocalOK:
    def allow(self, key): return True

for mode in ("fail_open", "fail_closed", "fail_local"):
    r = ResilientLimiter(BrokenShared(), LocalOK(), mode=mode)
    print(f"{mode}: allow() → {r.allow('alice')}")

## 5️⃣ Choosing the limit key (and the traps)

Whatever you count against — IP, user id, API key — attackers will try to
hide behind it. A quick tour of traps:

| Key | Good when | Gotcha |
|---|---|---|
| **Client IP (v4)** | Anonymous traffic | Office/school/NAT makes **many users share one IP** (you'll block them all). |
| **Client IP (v6)** | Anonymous traffic | A single customer gets a whole `/64` → `2^64` "different" IPs. Always limit on the **`/64` prefix**, not the full address. |
| **`X-Forwarded-For`** | Behind a proxy you control | Spoofable unless you strip untrusted hops. Only trust values from *your* load balancer. |
| **User id** | Logged-in users | Users can create many accounts — add a limit per email / device too. |
| **API key** | B2B / paid APIs | Best signal, but leaked keys get abused — support key rotation. |
| **Tenant id** | Multi-tenant SaaS | One noisy tenant can starve others — limit per tenant *and* globally. |

> 💡 Cheap trick: use a **tuple** as the key — `(tenant_id, user_id, route)`
> — and let the data structure pick the right granularity.


In [ ]:
import ipaddress

def limit_key_for(ip_str, route, user_id=None):
    """Pick a reasonable key for rate limiting."""
    ip = ipaddress.ip_address(ip_str)
    if isinstance(ip, ipaddress.IPv6Address):
        # collapse to /64 — otherwise attacker rotates within their /64
        ip_part = str(ipaddress.ip_network(f"{ip}/64", strict=False).network_address)
    else:
        ip_part = str(ip)
    # authenticated users get their own bucket; anonymous share per-ip
    identity = f"user:{user_id}" if user_id else f"ip:{ip_part}"
    return (identity, route)

print(limit_key_for("203.0.113.5",           "/search"))
print(limit_key_for("203.0.113.5",           "/search", user_id=42))
print(limit_key_for("2001:db8:abcd:12::ff",  "/search"))   # v6 → /64 prefix
print(limit_key_for("2001:db8:abcd:12::aa",  "/search"))   # same /64 → same key

## 6️⃣ Observability — you can't tune what you can't see

A limiter in production is invisible without metrics. Emit at least:

| Metric | Type | Why it matters |
|---|---|---|
| `rl_allowed_total{route}` | counter | Baseline traffic shape. |
| `rl_denied_total{route, reason}` | counter | Spot abuse & misconfig (`reason` = `rate` / `concurrency` / `cost`). |
| `rl_retry_after_seconds` | histogram | Are we telling clients reasonable wait times? |
| `rl_store_error_total{op}` | counter | **Fail-open failures are silent without this.** Alert on it. |
| Heavy-hitter **sample logs** | log / sketch | Who are the noisy keys? Don't label a metric with the key (high cardinality → $$). |

Pair those with a dashboard showing *denied ÷ (allowed + denied)* per route.
A sustained spike means either an attack, a misbehaving client, or a limit
that's too tight — all three are worth paging someone for.


In [ ]:
from collections import Counter

class ObservableLimiter:
    def __init__(self, inner):
        self.inner = inner
        self.metrics = Counter()          # pretend this is Prometheus
        self.top_keys_denied = Counter()  # sampled — bounded memory in real code

    def allow(self, key, route):
        try:
            ok = self.inner.allow(key)
        except Exception:
            self.metrics[("store_error", route)] += 1
            ok = True   # fail-open
        if ok:
            self.metrics[("allowed", route)] += 1
        else:
            self.metrics[("denied", route, "rate")] += 1
            self.top_keys_denied[key] += 1
        return ok

class Inner:
    def __init__(self): self.n = 0
    def allow(self, key):
        self.n += 1
        return self.n % 3 != 0   # fake: every 3rd request denied

obs = ObservableLimiter(Inner())
for i in range(20):
    obs.allow(f"user{i % 4}", route="/search")

print("metrics  :", dict(obs.metrics))
print("top noisy:", obs.top_keys_denied.most_common(3))


## ✅ Production checklist

- [ ] **Both** rate limit (arrivals) **and** concurrency limit (in-flight).
- [ ] Different limits for different endpoints (cost-aware).
- [ ] Return `429` + `Retry-After` + `X-RateLimit-*` headers.
- [ ] Pick the **key** carefully (IPv6 → `/64`, trust only your own proxies).
- [ ] Decide **fail-open or fail-closed** per endpoint and *test it*.
- [ ] **Observe**: log/emit metrics for `allowed`, `denied`, `limit`,
      `top keys` — you can't tune what you can't see.
- [ ] Document the limits in your API docs (developers *will* ask).

You now have enough to walk into a system-design interview and actually
design the rate-limiting layer of a real API. 👍
